In [1]:
from pyspark.sql.functions import *

In [ ]:
catalog = dbutils.widgets.get("catalog")
df = spark.read.table(f"{catalog}.02_silver.jc_citibike")

In [3]:
df.printSchema()

root
 |-- ride_id: string (nullable = true)
 |-- rideable_type: string (nullable = true)
 |-- started_at: timestamp (nullable = true)
 |-- ended_at: timestamp (nullable = true)
 |-- start_station_name: string (nullable = true)
 |-- start_station_id: string (nullable = true)
 |-- end_station_name: string (nullable = true)
 |-- end_station_id: string (nullable = true)
 |-- start_lat: decimal(10,0) (nullable = true)
 |-- start_lng: decimal(10,0) (nullable = true)
 |-- end_lat: decimal(10,0) (nullable = true)
 |-- end_lng: decimal(10,0) (nullable = true)
 |-- member_casual: string (nullable = true)
 |-- metadata: map (nullable = true)
 |    |-- key: string
 |    |-- value: string (valueContainsNull = true)
 |-- trip_duration_min: double (nullable = true)
 |-- trip_start_date: date (nullable = true)



In [4]:
df = df.groupBy(col("trip_start_date")).agg(
    round(max(col("trip_duration_min")),2).alias("max_trip_duration_in_mins"),
    round(min(col("trip_duration_min")),2).alias("min_trip_duration_in_mins"),
    round(avg(col("trip_duration_min")),2).alias("avg_trip_duration_in_mins"),
    count(col("ride_id")).alias("total_trips")
)

In [5]:
df.orderBy("trip_start_date").show()

+---------------+-------------------------+-------------------------+-------------------------+-----------+
|trip_start_date|max_trip_duration_in_mins|min_trip_duration_in_mins|avg_trip_duration_in_mins|total_trips|
+---------------+-------------------------+-------------------------+-------------------------+-----------+
|     2025-02-28|                    25.25|                      2.8|                     11.4|          4|
|     2025-03-01|                  1499.65|                     1.02|                    10.14|       2160|
|     2025-03-02|                  1499.93|                     1.05|                     12.0|       1147|
|     2025-03-03|                    138.7|                     1.02|                     6.53|       2030|
|     2025-03-04|                  1499.92|                     1.07|                     7.51|       2410|
|     2025-03-05|                    96.83|                     1.02|                     6.59|       1654|
|     2025-03-06|           

In [ ]:
df.write.mode("overwrite").option("overwriteSchema",True).saveAsTable(f"{catalog}.03_gold.daily_ride_summary")